In [13]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# **Model 3 — ELECTRA-base-discriminator (Pretrained, Fine-Tuned)**

## Our third main model — a **pretrained transformer**, fine-tuned using Hugging Face Transformers' `AutoModelForMultipleChoice`, treating each question as a 5-way multiple-choice classification problem.

## Install & Import Libraries 

In [14]:
!pip install transformers -q

import os
import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForMultipleChoice
from transformers import get_linear_schedule_with_warmup
from transformers import logging as hf_logging
from torch.optim import AdamW
import wandb

hf_logging.set_verbosity_error()

print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available())

PyTorch: 2.10.0+cu128
CUDA   : True


## Configuration


In [15]:
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
DATA_PATH   = "/kaggle/input/competitions/smart-mcq-solver-challenge"
MODEL_NAME  = "google/electra-base-discriminator"
OPTION_COLS = ["A", "B", "C", "D", "E"]
MAX_LEN     = 128
BATCH_SIZE  = 16
EPOCHS      = 3
LR          = 3e-5
SEED        = 42
VAL_SIZE    = 0.1

print("Device:", DEVICE)
print("Model :", MODEL_NAME)


Device: cuda
Model : google/electra-base-discriminator


## W&B Setup 

In [16]:
os.environ["WANDB_SILENT"] = "true"

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

wandb.init(
    project="smart-mcq-solver",
    name="model3-electra-base",
    config={
        "model_name" : MODEL_NAME,
        "max_len"    : MAX_LEN,
        "batch_size" : BATCH_SIZE,
        "epochs"     : EPOCHS,
        "lr"         : LR
    }
)


## Load Dataset 

In [17]:
train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

print("Train:", train.shape)
print("Test :", test.shape)
train.head(3)


Train: (2000, 8)
Test : (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


## **Train & Validation Split**

## Stratified splitting ensures the train and validation sets preserve the same class (answer option) proportions

In [18]:
torch.manual_seed(SEED)
np.random.seed(SEED)

train_df, val_df = train_test_split(
    train, test_size=VAL_SIZE, random_state=SEED, stratify=train['answer']
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)


Train: (1800, 8)
Val  : (200, 8)


## MAP@3 Utility 

In [19]:
def map_at_3(ground_truth, predictions):
    score = 0.0
    for k, pred in enumerate(predictions[:3], start=1):
        if pred == ground_truth:
            score = 1.0 / k
            break
    return score

def evaluate_map3(df, pred_col='prediction'):
    scores = []
    for _, row in df.iterrows():
        preds = row[pred_col].split()
        scores.append(map_at_3(row['answer'], preds))
    return np.mean(scores)

print("MAP@3 ready.")


MAP@3 ready.


# **Dataset Preparation**
## Each question is tokenized as 5 separate (prompt, option) pairs — this is the standard format expected by AutoModelForMultipleChoice.


## Tokenizer + Dataset Class

In [20]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128, is_test=False):
        self.df      = df.reset_index(drop=True)
        self.tok     = tokenizer
        self.max_len = max_len
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        prompt  = str(row['prompt'])
        options = [str(row[c]) for c in OPTION_COLS]

        encodings = []
        for opt in options:
            enc = self.tok(
                prompt, opt,
                max_length=self.max_len,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            encodings.append(enc)

        input_ids      = torch.cat([e['input_ids'] for e in encodings], dim=0)
        attention_mask = torch.cat([e['attention_mask'] for e in encodings], dim=0)
        out = {'input_ids': input_ids, 'attention_mask': attention_mask}

        if not self.is_test:
            out['labels'] = torch.tensor(OPTION_COLS.index(row['answer']), dtype=torch.long)
        return out


## Create Datasets & Loaders 

In [21]:
train_dataset = MCQDataset(train_df, tokenizer, MAX_LEN)
val_dataset   = MCQDataset(val_df,   tokenizer, MAX_LEN)
test_dataset  = MCQDataset(test,     tokenizer, MAX_LEN, is_test=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))


Train batches: 113
Val batches  : 13


## **Load Pretrained Model** 
## Loads google/electra-base-discriminator with a new multiple-choice classification head on top (randomly initialized, since it doesn't exist in the original pretrained checkpoint — this is what fine-tuning trains)

In [22]:
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
print("Model loaded:", MODEL_NAME)
print("Parameters  :", sum(p.numel() for p in model.parameters()) // 1_000_000, "M")


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Model loaded: google/electra-base-discriminator
Parameters  : 109 M


# **Training** 
## Optimizer & Scheduler Setup

In [23]:
optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

best_map3  = 0.0
best_epoch = 0


## Training Loop  

In [24]:
for epoch in range(1, EPOCHS + 1):

    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        logits  = outputs.logits

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        preds    = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
        total_loss += loss.item()

        if (batch_idx + 1) % 20 == 0:
            print(f"  Epoch {epoch} | Batch {batch_idx + 1}/{len(train_loader)} | "
                  f"Loss so far: {total_loss / (batch_idx + 1):.4f}")

    train_acc  = correct / total
    train_loss = total_loss / len(train_loader)

    model.eval()
    val_preds = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
            logits         = outputs.logits
            top3           = logits.argsort(dim=-1, descending=True)[:, :3]
            for row in top3:
                val_preds.append(' '.join([OPTION_COLS[i] for i in row.cpu().numpy()]))

    val_df['prediction'] = val_preds
    val_map3 = evaluate_map3(val_df)

    print(f"Epoch {epoch} | Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Val MAP@3: {val_map3:.4f}")

    wandb.log({
        "epoch"      : epoch,
        "train_loss" : round(train_loss, 5),
        "train_acc"  : round(train_acc,  5),
        "val_map3"   : round(val_map3,   5)
    })

    if val_map3 > best_map3:
        best_map3  = val_map3
        best_epoch = epoch
        print(f"Best epoch: {epoch} | MAP@3: {val_map3:.4f}")

print(f"Best Val MAP@3: {best_map3:.4f} at epoch {best_epoch}")
wandb.log({"best_val_map3": best_map3, "best_epoch": best_epoch})


  Epoch 1 | Batch 20/113 | Loss so far: 1.6085
  Epoch 1 | Batch 40/113 | Loss so far: 1.5902
  Epoch 1 | Batch 60/113 | Loss so far: 1.5349
  Epoch 1 | Batch 80/113 | Loss so far: 1.4449
  Epoch 1 | Batch 100/113 | Loss so far: 1.3494
Epoch 1 | Loss: 1.2999 | Acc: 0.5094 | Val MAP@3: 0.9258
Best epoch: 1 | MAP@3: 0.9258
  Epoch 2 | Batch 20/113 | Loss so far: 0.6422
  Epoch 2 | Batch 40/113 | Loss so far: 0.5856
  Epoch 2 | Batch 60/113 | Loss so far: 0.5330
  Epoch 2 | Batch 80/113 | Loss so far: 0.4870
  Epoch 2 | Batch 100/113 | Loss so far: 0.4479
Epoch 2 | Loss: 0.4410 | Acc: 0.8622 | Val MAP@3: 0.9975
Best epoch: 2 | MAP@3: 0.9975
  Epoch 3 | Batch 20/113 | Loss so far: 0.2441
  Epoch 3 | Batch 40/113 | Loss so far: 0.2186
  Epoch 3 | Batch 60/113 | Loss so far: 0.2049
  Epoch 3 | Batch 80/113 | Loss so far: 0.1998
  Epoch 3 | Batch 100/113 | Loss so far: 0.1899
Epoch 3 | Loss: 0.1936 | Acc: 0.9389 | Val MAP@3: 0.9975
Best Val MAP@3: 0.9975 at epoch 2


## Test Inference

In [25]:
model.eval()
test_preds = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
        logits         = outputs.logits
        top3           = logits.argsort(dim=-1, descending=True)[:, :3]
        for row in top3:
            test_preds.append(' '.join([OPTION_COLS[i] for i in row.cpu().numpy()]))

        if (batch_idx + 1) % 10 == 0:
            print(f"Processed batch {batch_idx + 1}/{len(test_loader)}")

print("Test inference complete.")


Processed batch 10/32
Processed batch 20/32
Processed batch 30/32
Test inference complete.


# Submission

In [26]:
submission = pd.DataFrame({
    'ID'         : test['id'],
    'Prediction' : test_preds
})

submission.to_csv('submission.csv', index=False)
print("Submission created.")
print(submission.head(10))

wandb.finish()


Submission created.
   ID Prediction
0   1      A D C
1   2      B D C
2   3      B E D
3   4      E C D
4   5      C A D
5   6      D A E
6   7      E D C
7   8      B C E
8   9      C D E
9  10      B E C


## **Summary**

| Metric | Value |
|---|---|
| Model | ELECTRA-base-discriminator |
| Category | Pretrained transformer (fine-tuned) |
| Training required | Yes (fine-tuning only; base weights pretrained) |
| Kaggle MAP@3 | 0.75519 |

This is the highest-scoring model of the three, demonstrating the strength
of transfer learning — the pretrained language understanding from
`google/electra-base-discriminator` is fine-tuned on this competition's
multiple-choice task using a new classification head.
